In [1]:
print('Đang khai báo thư viện')

import joblib
import pandas as pd
import os
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [5]:
model_path = os.path.join("..", "Model", "ensemble_stack_final.pkl")
feature_path = os.path.join("..", "Model", "feature_names_ensemble.pkl")

model = joblib.load(model_path)
feature_names = joblib.load(feature_path)

print(f"Đã load Ensemble model")
print(f"Features ({len(feature_names)}): {feature_names}")

Đã load Ensemble model
Features (15): ['Popularity', 'danceability', 'energy', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'key_sin', 'key_cos']


In [6]:

test_path = os.path.join("..", "Data", "DataCleaned", "test_cleaned.csv")
df_test = pd.read_csv(test_path)
print(f"\nTest shape: {df_test.shape}")

drop_cols = ['Id', 'Artist Name', 'Track Name']
existing_drop_test = [col for col in drop_cols if col in df_test.columns]

if 'Class' in df_test.columns:
    X_test = df_test.drop(columns=['Class'] + existing_drop_test)
    y_test = df_test['Class']
    has_label = True
else:
    X_test = df_test.drop(columns=existing_drop_test)
    has_label = False

if 'key' in X_test.columns:
    X_test = X_test.drop(columns=['key'])
    print("Đã bỏ feature 'key' khỏi test set")

X_test = X_test[feature_names]
print(f"X_test shape sau khi align features: {X_test.shape}")


Test shape: (3600, 16)
Đã bỏ feature 'key' khỏi test set
X_test shape sau khi align features: (3600, 15)


In [7]:
y_pred_test = model.predict(X_test)

print("\n=== KẾT QUẢ DỰ ĐOÁN TRÊN TEST ===")
print(f"Số lượng mẫu test: {len(y_pred_test)}")
print(f"Phân bố class dự đoán:\n{pd.Series(y_pred_test).value_counts().sort_index()}")

if has_label:
    print(f"\nAccuracy Test : {accuracy_score(y_test, y_pred_test):.4f}")
    print(f"Macro F1 Test : {f1_score(y_test, y_pred_test, average='macro', zero_division=0):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_test, zero_division=0))

submission = pd.DataFrame({
    'Id': df_test['Id'] if 'Id' in df_test.columns else range(len(y_pred_test)),
    'Class': y_pred_test
})
submission.to_csv('submission_ensemble.csv', index=False)
print("\nĐã lưu kết quả vào submission_ensemble.csv")


=== KẾT QUẢ DỰ ĐOÁN TRÊN TEST ===
Số lượng mẫu test: 3600
Phân bố class dự đoán:
0     285
1     324
2     363
3     125
4     254
5     295
6     361
7     134
8     539
9     451
10    469
Name: count, dtype: int64

Đã lưu kết quả vào submission_ensemble.csv


In [8]:
sample_path = os.path.join("..", "Data", "sample_submission.csv")
sample = pd.read_csv(sample_path)

print(f"Sample shape: {sample.shape}")
print(f"Sample Id range: {sample['Id'].min()} - {sample['Id'].max()}")
print(f"First 10 Ids: {sample['Id'].head(10).tolist()}")

correct_ids = sample['Id'].values

print(f"Number of predictions: {len(y_pred_test)}")

if len(correct_ids) != len(y_pred_test):
    print(f"Warning: Sample IDs ({len(correct_ids)}) != Predictions ({len(y_pred_test)})")
    min_len = min(len(correct_ids), len(y_pred_test))
    correct_ids = correct_ids[:min_len]
    y_pred_adjusted = y_pred_test[:min_len]
else:
    y_pred_adjusted = y_pred_test

submission_fixed = pd.DataFrame({
    'Id': correct_ids,
    'Class': y_pred_adjusted
})

output_dir = os.path.join("..", "Submissions")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "submission_ensemble.csv")
submission_fixed.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
print(f"Submission shape: {submission_fixed.shape}")
print(f"Id range: {submission_fixed['Id'].min()} - {submission_fixed['Id'].max()}")

# Kiểm tra nhanh phân bố class trong submission
print(f"\nClass distribution in submission:")
print(submission_fixed['Class'].value_counts().sort_index())

print("\nFile ready for Kaggle submission!")

Sample shape: (3600, 2)
Sample Id range: 14397 - 17996
First 10 Ids: [14397, 14398, 14399, 14400, 14401, 14402, 14403, 14404, 14405, 14406]
Number of predictions: 3600

Saved: ..\Submissions\submission_ensemble.csv
Submission shape: (3600, 2)
Id range: 14397 - 17996

Class distribution in submission:
Class
0     285
1     324
2     363
3     125
4     254
5     295
6     361
7     134
8     539
9     451
10    469
Name: count, dtype: int64

File ready for Kaggle submission!
